In [ ]:
# ══════════════════════════════════════════════════════════════════
# Google Colab setup — clones YOUR GitHub Classroom fork so that
# `from shared.mlfp05.diagnostics import ...` resolves correctly.
# ══════════════════════════════════════════════════════════════════
import os, sys

# ① EDIT THIS to point at YOUR fork of the Classroom repo.
#    Your fork URL is at the top of your assignment page on GitHub.
#    Example: "https://github.com/janedoe/pcml-run26-2601-janedoe.git"
FORK_URL = "https://github.com/<your-github-username>/<your-fork>.git"
REPO_DIR = "/content/pcml-run26"

if not os.path.exists(REPO_DIR):
    !git clone {FORK_URL} {REPO_DIR}

# ② cd into the repo so relative data paths resolve
%cd {REPO_DIR}

# ③ Install deps (most are pre-installed on Colab)
!pip install -q polars plotly gdown python-dotenv

# ④ Make the `shared` package importable
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ⑤ (Optional) Mount Drive if your exercise reads from Drive
# from google.colab import drive
# drive.mount("/content/drive")

print("✓ Colab setup complete — shared.mlfp05 is importable")


# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 1.4: Sparse Autoencoder (Sparsity Penalty)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Build a sparse AE with L1 penalty on hidden activations
#   - Understand WHY sparsity forces specialist neurons
#   - Visualise activation histograms proving sparsity
#   - Apply to semiconductor wafer defect detection at GlobalFoundries
#   - Quantify business impact: defect detection rate + cost savings
#
# PREREQUISITES: 03_denoising_ae.py
# ESTIMATED TIME: ~20 min
#
# TASKS:
#   1. Build Sparse AE with L1 regularisation
#   2. Train and verify sparsity in activation distribution
#   3. Visualise reconstruction + activation histograms
#   4. Apply: wafer defect detection with error heatmaps
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from shared.mlfp05.ex_1 import (
    INPUT_DIM,
    EPOCHS,
    OUTPUT_DIR,
    device,
    load_fashion_mnist,
    setup_engines,
    train_variant,
    show_reconstruction,
    show_activation_sparsity,
    register_model,
)



## THEORY — Sparsity Forces Specialist Neurons


In [ ]:
# Instead of constraining the bottleneck SIZE, we constrain the
# ACTIVATIONS. An L1 penalty on the hidden layer forces most neurons
# to stay near zero. Only a few "specialist" neurons fire for each input.
#
# Analogy: In a bank's fraud team, you want each analyst to specialise.
# One analyst for "unusual amounts", another for "unusual times",
# another for "unusual merchants". Sparsity forces this specialisation.
# The model cannot spread the signal across all neurons equally.
#
# Biological parallel: In the visual cortex, only ~1% of neurons fire
# for any given stimulus. Sparse representations are more interpretable
# and more energy-efficient — nature discovered this long before ML.



## TASK 1 — Load data and engines


In [ ]:
X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader = load_fashion_mnist()
conn, tracker, exp_name, registry, has_registry = setup_engines()



## TASK 2 — Build and Train Sparse AE


MSE + L1 sparsity penalty on hidden activations.


In [ ]:
class SparseAE(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 256):
        super().__init__()
        # TODO: Build encoder — nn.Sequential:
        #       Linear(input_dim, hidden_dim), ReLU,
        #       Linear(hidden_dim, 128), ReLU,
        #       Linear(128, 64)
        self.encoder = ____

        # TODO: Build decoder — mirror:
        #       Linear(64, 128), ReLU, Linear(128, hidden_dim), ReLU,
        #       Linear(hidden_dim, input_dim), Sigmoid
        self.decoder = ____

    def forward(self, x):
        # TODO: Encode then decode. Return (reconstruction, latent_code)
        ____


SPARSITY_WEIGHT = 1e-4


def sparse_ae_loss(model, xb):
    # TODO: Forward pass to get x_hat and z
    # recon_loss = F.mse_loss(x_hat, xb)
    # sparsity_loss = SPARSITY_WEIGHT * torch.mean(torch.abs(z))
    # Return (recon_loss + sparsity_loss, {"sparsity": sparsity_loss.item()})
    ____


print("\n" + "=" * 70)
print("  Sparse AE — L1 Sparsity Penalty")
print("=" * 70)
print(f"  Sparsity weight: {SPARSITY_WEIGHT}. Most neurons should stay near zero.")

# TODO: Create SparseAE(INPUT_DIM) and train
sparse_model = ____
sparse_losses = ____



## TASK 3 — Visualise Reconstruction + Sparsity


In [ ]:
# TODO: show_reconstruction and show_activation_sparsity
____
____



### Checkpoint


In [ ]:
assert len(sparse_losses) == EPOCHS
assert sparse_losses[-1] < sparse_losses[0]
sparse_model.eval()
with torch.no_grad():
    test_z = sparse_model.encoder(X_test_flat[:1000].to(device))
    pct_sparse = (torch.abs(test_z) < 0.1).float().mean().item()
assert pct_sparse > 0.3, f"Expected >30% activations near zero, got {pct_sparse:.1%}"
print(f"  Sparsity: {pct_sparse:.1%} of activations near zero")
print("\n--- Checkpoint passed --- sparse AE trained\n")

if has_registry:
    register_model(registry, "sparse_ae", sparse_model, sparse_losses[-1])



## APPLY — Semiconductor Wafer Defect Detection (GlobalFoundries)


In [ ]:
# BUSINESS SCENARIO: You are an ML engineer at a semiconductor fab in
# Singapore (GlobalFoundries/SSMC). Visual inspection of silicon wafers
# is the quality bottleneck — manual inspection catches 82% of defects
# at 15 seconds per wafer. A missed defect costs S$5,000 in downstream
# rework. Your plant manager asks: "Can we automate inspection?"

print("\n" + "=" * 70)
print("  APPLICATION: Wafer Defect Detection (GlobalFoundries)")
print("=" * 70)

# --- Generate synthetic wafer images ---
IMG_SIZE = 64
N_GOOD = 3000
N_DEFECTIVE = 400
wafer_rng = np.random.default_rng(42)


# TODO: Implement generate_wafer_base(rng_local) -> np.ndarray (64x64)
# Create a circular wafer mask, add die grid pattern inside, add edge ring,
# add small noise. Return clipped [0, 1] image.
def generate_wafer_base(rng_local):
    ____


# TODO: Implement generate_defective_wafer(rng_local) -> (image, mask, defect_type)
# Start from base wafer, add one of: scratch, particle, edge_chip, hotspot
# Return (clipped_image, defect_location_mask, defect_type_string)
def generate_defective_wafer(rng_local):
    ____


# TODO: Generate good_wafers and defective data
good_wafers = ____
defect_data = ____
defective_wafers = np.stack([d[0] for d in defect_data])
defect_masks = np.stack([d[1] for d in defect_data])
defect_types = [d[2] for d in defect_data]

n_train_wafer = int(N_GOOD * 0.8)
train_good_tensor = torch.tensor(good_wafers[:n_train_wafer, None, :, :], device=device)
test_good_tensor = torch.tensor(good_wafers[n_train_wafer:, None, :, :], device=device)
test_defective_tensor = torch.tensor(defective_wafers[:, None, :, :], device=device)
wafer_train_loader = DataLoader(
    TensorDataset(train_good_tensor), batch_size=64, shuffle=True
)

print(f"Good wafers: {N_GOOD}, Defective: {N_DEFECTIVE}")


class SparseConvAE(nn.Module):
    def __init__(self, sparsity_weight: float = 1e-3):
        super().__init__()
        self.sparsity_weight = sparsity_weight
        # TODO: Build encoder — 3 Conv2d layers:
        #       Conv2d(1, 16, 3, stride=2, padding=1), ReLU,
        #       Conv2d(16, 32, 3, stride=2, padding=1), ReLU,
        #       Conv2d(32, 64, 3, stride=2, padding=1), ReLU
        self.encoder = ____

        # TODO: Build decoder — 3 ConvTranspose2d layers:
        #       ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), ReLU,
        #       ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1), ReLU,
        #       ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1), Sigmoid
        self.decoder = ____

    def forward(self, x):
        # TODO: Encode then decode. Return (reconstruction, latent)
        ____

    def loss(self, recon, x, z):
        # TODO: MSE + sparsity_weight * mean(abs(z))
        ____


# TODO: Create model, optimizer. Train 50 epochs on good wafers only.
wafer_model = ____
wafer_opt = ____

print("\nTraining sparse autoencoder on good wafers...")
for epoch in range(50):
    wafer_model.train()
    epoch_loss, n_batches = 0.0, 0
    for (batch,) in wafer_train_loader:
        # TODO: Forward, compute loss with wafer_model.loss(), backprop
        ____
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d}/50: loss = {epoch_loss/n_batches:.6f}")

# --- Compute reconstruction errors ---
wafer_model.eval()
with torch.no_grad():
    recon_good, _ = wafer_model(test_good_tensor)
    recon_defective, _ = wafer_model(test_defective_tensor)
    good_errors = (
        ((test_good_tensor - recon_good) ** 2).mean(dim=(1, 2, 3)).cpu().numpy()
    )
    defective_errors = (
        ((test_defective_tensor - recon_defective) ** 2)
        .mean(dim=(1, 2, 3))
        .cpu()
        .numpy()
    )
    pixel_errors_defective = (
        ((test_defective_tensor - recon_defective) ** 2).squeeze(1).cpu().numpy()
    )

print(
    f"\nReconstruction errors: Good={good_errors.mean():.6f}, Defective={defective_errors.mean():.6f}"
)
print(f"Separation: {defective_errors.mean() / good_errors.mean():.1f}x")

# --- Visualisation: Defect localisation ---
# TODO: Create 4x5 grid showing: input, reconstructed, error heatmap, ground truth
# for 5 defective wafers. Save to OUTPUT_DIR / "ex1_manufacturing_defect_localisation.png"
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
____
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ex1_manufacturing_defect_localisation.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

# --- Business Impact ---
WAFERS_PER_DAY = 5000
MANUAL_DETECTION_RATE = 0.82
COST_PER_MISSED_DEFECT = 5000
DEFECT_RATE = 0.03
INSPECTOR_ANNUAL_COST = 60_000
INSPECTORS_NEEDED = 4
WORKING_DAYS = 260

# TODO: Compute detection threshold and business savings
# Use the defective error distribution to find operating detection rate
all_wafer_errors = np.concatenate([good_errors, defective_errors])
all_wafer_labels = np.concatenate(
    [np.zeros(len(good_errors)), np.ones(len(defective_errors))]
)
thresholds = np.linspace(
    all_wafer_errors.min(), np.percentile(all_wafer_errors, 99.5), 200
)
detection_rates, false_alarm_rates = [], []
for t in thresholds:
    pred = all_wafer_errors > t
    tp = np.sum(pred & (all_wafer_labels == 1))
    fp = np.sum(pred & (all_wafer_labels == 0))
    fn = np.sum(~pred & (all_wafer_labels == 1))
    tn = np.sum(~pred & (all_wafer_labels == 0))
    detection_rates.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
    false_alarm_rates.append(fp / (fp + tn) if (fp + tn) > 0 else 0.0)

detection_rates = np.array(detection_rates)
false_alarm_rates = np.array(false_alarm_rates)
target_dr = 0.94
best_idx = np.argmin(np.abs(detection_rates - target_dr))
operating_dr = detection_rates[best_idx]

# TODO: Calculate annual savings from missed defects + labour
daily_defective = int(WAFERS_PER_DAY * DEFECT_RATE)
manual_missed = daily_defective - int(daily_defective * MANUAL_DETECTION_RATE)
ae_missed = daily_defective - int(daily_defective * operating_dr)
annual_missed_manual = ____
annual_missed_ae = ____
annual_savings_defects = ____
annual_savings_labour = ____
total_annual_savings = ____

print("\n" + "=" * 64)
print("BUSINESS IMPACT SUMMARY — Semiconductor Wafer Inspection")
print("=" * 64)
print(f"\nPlant capacity: {WAFERS_PER_DAY:,} wafers/day, {DEFECT_RATE:.0%} defect rate")
print(f"Manual detection rate:   {MANUAL_DETECTION_RATE:>8.0%}")
print(f"Sparse AE detection:     {operating_dr:>8.0%}")
print(f"\nAnnual financial impact:")
print(f"  Missed defect savings:        {'S$' + f'{annual_savings_defects:,.0f}':>14}")
print(f"  Labour savings (3 FTE):       {'S$' + f'{annual_savings_labour:,.0f}':>14}")
print(f"  Total annual savings:         {'S$' + f'{total_annual_savings:,.0f}':>14}")
print("=" * 64)



## REFLECTION


[x] Built a sparse AE with L1 penalty on hidden activations
  [x] Verified sparsity: >30% of activations near zero
  [x] Visualised activation histogram proving specialist neurons
  [x] Applied to semiconductor wafer defect detection
  [x] Generated error heatmaps that localise defects spatially
  [x] Quantified business impact: detection rate + S$ savings

  KEY INSIGHT: Sparsity forces each neuron to specialise. Instead
  of all neurons contributing a little to every reconstruction,
  a few "expert" neurons fire for each specific pattern. This makes
  the representation both interpretable and efficient — exactly
  what you need for defect detection where you want to know
  WHICH feature detected WHICH defect.

  Next: 05_contractive_ae.py smooths the latent space with Jacobian penalty...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — five instruments (sparsity under X-ray)


In [ ]:
# The L1 penalty FORCES most neurons to zero. The X-ray should show
# high dead_fraction on the sparse layer — but unlike 01's PATHOLOGICAL
# dead-ReLU failure, here dead-neuron-ness is the DESIGN GOAL.
# Students must learn to read intent from the report, not just icons.
from shared.mlfp05.diagnostics import run_diagnostic_checkpoint


def _diag_loss(m, batch):
    xb = batch[0] if isinstance(batch, (tuple, list)) else batch
    loss, _ = sparse_ae_loss(m, xb)
    return loss


print("\n── Diagnostic Report (Sparse AE) ──")
diag, findings = run_diagnostic_checkpoint(
    sparse_model,
    flat_loader,
    _diag_loss,
    title=f"Sparse AE (L1={SPARSITY_WEIGHT})",
    n_batches=8,
    train_losses=sparse_losses,
    show=False,
)

# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
#   [!] Dead neurons  (WARNING): 'encoder.3' (relu): 87% dead
#       neurons.  (BY DESIGN — L1 penalty active)
#       Fix: WARNING suppressable — see interpretation guide.
#   [✓] Gradient flow (HEALTHY): min RMS = 2.1e-04 at
#       'encoder.3.weight'. The 13% of channels that are ACTIVE
#       carry the reconstruction — exactly the sparse-coding goal.
#   [✓] Loss trend    (HEALTHY): train slope -1.6e-03/epoch.
#       Final loss = reconstruction (~0.013) + L1 penalty (~0.003).



## Final train loss: ~0.016 after 10 epochs, L1 weight=1e-3.

STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:

 [X-RAY — CRITICAL REFRAMING] 87% dead neurons is the TARGET,
    not the bug. L1 regularisation drives most neurons to zero
    FOR EACH INPUT so that the remaining 13% encode a specific
    concept. This is precisely the sparse-coding objective from
    Olshausen & Field (1996) that slide 5H walks through —
    unlike 01's 59% which was ACCIDENTAL (ReLU dying), here
    87% is DESIGNED (L1 winning per-sample).
    >> Prescription: IGNORE the WARNING. Verify instead via
       `diag.activations_df()` that the INACTIVE FRACTION per
       BATCH matches your target (~90% here). If it exceeds
       95%, L1 is too strong — halve SPARSITY_WEIGHT.

 [BLOOD TEST] Gradient RMS 2.1e-04 at encoder.3 is healthy.
    Contrast with 01's 9.46e-06 at decoder.2. Even though most
    neurons are dead, the ACTIVE ones receive strong, focused
    gradients — concentrated learning.
    >> Prescription: If RMS drops below 1e-5, L1 is killing
       every neuron PERMANENTLY (not per-sample). Lower
       SPARSITY_WEIGHT to 5e-4 and retrain.

 [STETHOSCOPE] Loss trend healthy with a two-term composition:
    reconstruction MSE + L1 penalty. A rising L1 curve while
    MSE falls signals the regulariser hasn't converged yet —
    train longer. A rising L1 AND MSE signals overshoot of
    the regulariser.
    >> Prescription: Plot `diag.epochs_df()` split into
       `recon_loss` vs `l1_loss` columns. Both should decay.

 FIVE-INSTRUMENT TAKEAWAY: this file teaches the CLINICAL
 READING SKILL. Same X-Ray finding (high % dead), opposite
 diagnosis (01: kill the model / 04: ship the model). You
 will revisit this skill in 05_contractive (different
 regulariser, same "context determines pathology" principle)
 and again in ex_6 GNN over-smoothing (where "similar
 embeddings" is bug in shallow nets, feature in deep nets).


## TASK 3 — Visualise Reconstruction + Sparsity


In [ ]:
show_reconstruction(sparse_model, X_test_flat, "Sparse AE")
show_activation_sparsity(sparse_model, X_test_flat, "Sparse AE — Hidden Activations")



### Checkpoint


In [ ]:
assert len(sparse_losses) == EPOCHS
assert sparse_losses[-1] < sparse_losses[0]
sparse_model.eval()
with torch.no_grad():
    test_z = sparse_model.encoder(X_test_flat[:1000].to(device))
    pct_sparse = (torch.abs(test_z) < 0.1).float().mean().item()
assert pct_sparse > 0.3, f"Expected >30% activations near zero, got {pct_sparse:.1%}"
print(f"  Sparsity: {pct_sparse:.1%} of activations near zero")
print("\n--- Checkpoint passed --- sparse AE trained\n")

if has_registry:
    register_model(registry, "sparse_ae", sparse_model, sparse_losses[-1])



## APPLY — Semiconductor Wafer Defect Detection (GlobalFoundries)


In [ ]:
# BUSINESS SCENARIO: You are an ML engineer at a semiconductor fab in
# Singapore (GlobalFoundries/SSMC). Visual inspection of silicon wafers
# is the quality bottleneck — manual inspection catches 82% of defects
# at 15 seconds per wafer. A missed defect costs S$5,000 in downstream
# rework. Your plant manager asks: "Can we automate inspection?"

print("\n" + "=" * 70)
print("  APPLICATION: Wafer Defect Detection (GlobalFoundries)")
print("=" * 70)

# --- Generate synthetic wafer images ---
IMG_SIZE = 64
N_GOOD = 3000
N_DEFECTIVE = 400
wafer_rng = np.random.default_rng(42)


def generate_wafer_base(rng_local):
    img = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)
    yy, xx = np.mgrid[:IMG_SIZE, :IMG_SIZE]
    center = IMG_SIZE / 2
    radius = IMG_SIZE * 0.42
    wafer_mask = ((xx - center) ** 2 + (yy - center) ** 2) < radius**2
    img[wafer_mask] = 0.3 + rng_local.uniform(-0.02, 0.02)
    die_size, die_gap = 6, 1
    for dy in range(5, IMG_SIZE - 5, die_size + die_gap):
        for dx in range(5, IMG_SIZE - 5, die_size + die_gap):
            die_cx, die_cy = dx + die_size / 2, dy + die_size / 2
            if ((die_cx - center) ** 2 + (die_cy - center) ** 2) < (radius - 3) ** 2:
                img[dy : dy + die_size, dx : dx + die_size] = 0.6 + rng_local.uniform(
                    -0.03, 0.03
                )
    edge_mask = (
        ((xx - center) ** 2 + (yy - center) ** 2) > (radius - 2) ** 2
    ) & wafer_mask
    img[edge_mask] = 0.25
    img += rng_local.normal(0, 0.01, (IMG_SIZE, IMG_SIZE)).astype(np.float32)
    return np.clip(img, 0, 1)


def generate_defective_wafer(rng_local):
    img = generate_wafer_base(rng_local)
    mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)
    defect_type = rng_local.choice(["scratch", "particle", "edge_chip", "hotspot"])
    center = IMG_SIZE / 2
    yy, xx = np.mgrid[:IMG_SIZE, :IMG_SIZE]

    if defect_type == "scratch":
        angle = rng_local.uniform(0, np.pi)
        length = rng_local.integers(20, 45)
        cx, cy = center + rng_local.uniform(-10, 10), center + rng_local.uniform(
            -10, 10
        )
        for t_val in np.linspace(-length / 2, length / 2, length * 3):
            px, py = int(cx + t_val * np.cos(angle)), int(cy + t_val * np.sin(angle))
            if 0 <= px < IMG_SIZE and 0 <= py < IMG_SIZE:
                w = rng_local.integers(1, 3)
                img[
                    max(0, py - w) : min(IMG_SIZE, py + w),
                    max(0, px - w) : min(IMG_SIZE, px + w),
                ] = rng_local.uniform(0.7, 0.9)
                mask[
                    max(0, py - w) : min(IMG_SIZE, py + w),
                    max(0, px - w) : min(IMG_SIZE, px + w),
                ] = 1.0
    elif defect_type == "particle":
        for _ in range(rng_local.integers(3, 10)):
            px, py = rng_local.integers(10, IMG_SIZE - 10), rng_local.integers(
                10, IMG_SIZE - 10
            )
            r = rng_local.integers(1, 4)
            dist = np.sqrt((xx - px) ** 2 + (yy - py) ** 2)
            particle_mask = dist < r
            img[particle_mask] = rng_local.uniform(0.75, 0.95)
            mask[particle_mask] = 1.0
    elif defect_type == "edge_chip":
        angle = rng_local.uniform(0, 2 * np.pi)
        radius = IMG_SIZE * 0.42
        ex, ey = center + radius * np.cos(angle), center + radius * np.sin(angle)
        chip_r = rng_local.integers(4, 10)
        dist = np.sqrt((xx - ex) ** 2 + (yy - ey) ** 2)
        img[dist < chip_r] = rng_local.uniform(0.05, 0.15)
        mask[dist < chip_r] = 1.0
    elif defect_type == "hotspot":
        hx, hy = center + rng_local.uniform(-15, 15), center + rng_local.uniform(
            -15, 15
        )
        hr = rng_local.integers(5, 12)
        dist = np.sqrt((xx - hx) ** 2 + (yy - hy) ** 2)
        blob = np.exp(-(dist**2) / (2 * (hr / 2) ** 2)) * rng_local.uniform(0.2, 0.4)
        img -= blob.astype(np.float32)
        mask[dist < hr] = 1.0

    return np.clip(img, 0, 1), mask, defect_type


good_wafers = np.stack([generate_wafer_base(wafer_rng) for _ in range(N_GOOD)])
defect_data = [generate_defective_wafer(wafer_rng) for _ in range(N_DEFECTIVE)]
defective_wafers = np.stack([d[0] for d in defect_data])
defect_masks = np.stack([d[1] for d in defect_data])
defect_types = [d[2] for d in defect_data]

n_train_wafer = int(N_GOOD * 0.8)
train_good_tensor = torch.tensor(good_wafers[:n_train_wafer, None, :, :], device=device)
test_good_tensor = torch.tensor(good_wafers[n_train_wafer:, None, :, :], device=device)
test_defective_tensor = torch.tensor(defective_wafers[:, None, :, :], device=device)
wafer_train_loader = DataLoader(
    TensorDataset(train_good_tensor), batch_size=64, shuffle=True
)

print(f"Good wafers: {N_GOOD}, Defective: {N_DEFECTIVE}")


class SparseConvAE(nn.Module):
    def __init__(self, sparsity_weight: float = 1e-3):
        super().__init__()
        self.sparsity_weight = sparsity_weight
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

    def loss(self, recon, x, z):
        mse = F.mse_loss(recon, x)
        sparsity = self.sparsity_weight * torch.mean(torch.abs(z))
        return mse + sparsity


wafer_model = SparseConvAE(sparsity_weight=1e-3).to(device)
wafer_opt = torch.optim.Adam(wafer_model.parameters(), lr=1e-3)

print("\nTraining sparse autoencoder on good wafers...")
for epoch in range(50):
    wafer_model.train()
    epoch_loss, n_batches = 0.0, 0
    for (batch,) in wafer_train_loader:
        recon, z = wafer_model(batch)
        loss = wafer_model.loss(recon, batch, z)
        wafer_opt.zero_grad()
        loss.backward()
        wafer_opt.step()
        epoch_loss += loss.item()
        n_batches += 1
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d}/50: loss = {epoch_loss/n_batches:.6f}")

# --- Compute reconstruction errors ---
wafer_model.eval()
with torch.no_grad():
    recon_good, _ = wafer_model(test_good_tensor)
    recon_defective, _ = wafer_model(test_defective_tensor)
    good_errors = (
        ((test_good_tensor - recon_good) ** 2).mean(dim=(1, 2, 3)).cpu().numpy()
    )
    defective_errors = (
        ((test_defective_tensor - recon_defective) ** 2)
        .mean(dim=(1, 2, 3))
        .cpu()
        .numpy()
    )
    pixel_errors_defective = (
        ((test_defective_tensor - recon_defective) ** 2).squeeze(1).cpu().numpy()
    )

print(
    f"\nReconstruction errors: Good={good_errors.mean():.6f}, Defective={defective_errors.mean():.6f}"
)
print(f"Separation: {defective_errors.mean() / good_errors.mean():.1f}x")

# --- Visualisation: Defect localisation ---
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
for i in range(5):
    axes[0, i].imshow(defective_wafers[i], cmap="gray", vmin=0, vmax=1)
    axes[0, i].set_title(f"Defective ({defect_types[i]})", fontsize=10)
    axes[0, i].axis("off")
    axes[1, i].imshow(recon_defective[i, 0].cpu().numpy(), cmap="gray", vmin=0, vmax=1)
    axes[1, i].set_title("Reconstructed", fontsize=10)
    axes[1, i].axis("off")
    axes[2, i].imshow(pixel_errors_defective[i], cmap="hot")
    axes[2, i].set_title(f"Error (MSE={defective_errors[i]:.5f})", fontsize=9)
    axes[2, i].axis("off")
    axes[3, i].imshow(defect_masks[i], cmap="hot", vmin=0, vmax=1)
    axes[3, i].set_title("Ground Truth", fontsize=10)
    axes[3, i].axis("off")
axes[0, 0].set_ylabel("Input", fontsize=12, rotation=0, labelpad=55)
axes[1, 0].set_ylabel("Recon", fontsize=12, rotation=0, labelpad=55)
axes[2, 0].set_ylabel("Error", fontsize=12, rotation=0, labelpad=55)
axes[3, 0].set_ylabel("Truth", fontsize=12, rotation=0, labelpad=55)
fig.suptitle(
    "Sparse AE: Wafer Defect Localisation\nError heatmaps pinpoint defect location",
    fontsize=13,
)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ex1_manufacturing_defect_localisation.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

# --- Detection curve ---
all_wafer_errors = np.concatenate([good_errors, defective_errors])
all_wafer_labels = np.concatenate(
    [np.zeros(len(good_errors)), np.ones(len(defective_errors))]
)
thresholds = np.linspace(
    all_wafer_errors.min(), np.percentile(all_wafer_errors, 99.5), 200
)
detection_rates, false_alarm_rates = [], []
for t in thresholds:
    pred = all_wafer_errors > t
    tp = np.sum(pred & (all_wafer_labels == 1))
    fp = np.sum(pred & (all_wafer_labels == 0))
    fn = np.sum(~pred & (all_wafer_labels == 1))
    tn = np.sum(~pred & (all_wafer_labels == 0))
    detection_rates.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
    false_alarm_rates.append(fp / (fp + tn) if (fp + tn) > 0 else 0.0)

detection_rates = np.array(detection_rates)
false_alarm_rates = np.array(false_alarm_rates)
target_dr = 0.94
best_idx = np.argmin(np.abs(detection_rates - target_dr))
operating_dr = detection_rates[best_idx]
operating_far = false_alarm_rates[best_idx]

# --- Business Impact ---
WAFERS_PER_DAY = 5000
MANUAL_DETECTION_RATE = 0.82
COST_PER_MISSED_DEFECT = 5000
DEFECT_RATE = 0.03
INSPECTOR_ANNUAL_COST = 60_000
INSPECTORS_NEEDED = 4
WORKING_DAYS = 260

daily_defective = int(WAFERS_PER_DAY * DEFECT_RATE)
manual_missed = daily_defective - int(daily_defective * MANUAL_DETECTION_RATE)
ae_missed = daily_defective - int(daily_defective * operating_dr)
annual_missed_manual = manual_missed * COST_PER_MISSED_DEFECT * WORKING_DAYS
annual_missed_ae = ae_missed * COST_PER_MISSED_DEFECT * WORKING_DAYS
annual_savings_defects = annual_missed_manual - annual_missed_ae
annual_savings_labour = (INSPECTORS_NEEDED - 1) * INSPECTOR_ANNUAL_COST
total_annual_savings = annual_savings_defects + annual_savings_labour

print("\n" + "=" * 64)
print("BUSINESS IMPACT SUMMARY — Semiconductor Wafer Inspection")
print("=" * 64)
print(f"\nPlant capacity: {WAFERS_PER_DAY:,} wafers/day, {DEFECT_RATE:.0%} defect rate")
print(f"Manual detection rate:   {MANUAL_DETECTION_RATE:>8.0%}")
print(f"Sparse AE detection:     {operating_dr:>8.0%}")
print(f"\nAnnual financial impact:")
print(f"  Missed defect savings:        {'S$' + f'{annual_savings_defects:,.0f}':>14}")
print(f"  Labour savings (3 FTE):       {'S$' + f'{annual_savings_labour:,.0f}':>14}")
print(f"  Total annual savings:         {'S$' + f'{total_annual_savings:,.0f}':>14}")
print("=" * 64)



## REFLECTION


[x] Built a sparse AE with L1 penalty on hidden activations
  [x] Verified sparsity: >30% of activations near zero
  [x] Visualised activation histogram proving specialist neurons
  [x] Applied to semiconductor wafer defect detection
  [x] Generated error heatmaps that localise defects spatially
  [x] Quantified business impact: detection rate + S$ savings

  KEY INSIGHT: Sparsity forces each neuron to specialise. Instead
  of all neurons contributing a little to every reconstruction,
  a few "expert" neurons fire for each specific pattern. This makes
  the representation both interpretable and efficient — exactly
  what you need for defect detection where you want to know
  WHICH feature detected WHICH defect.

  Next: 05_contractive_ae.py smooths the latent space with Jacobian penalty...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

